# Scope Drift Analysis Pipeline

1. **cwts_export.py** — Fetch citation network, run CWTS clustering, upload to BigQuery
2. **label_clusters** — Generate GPT labels for clusters
3. **build_unified_dashboard.py** — Create combined dashboard from BigQuery data

In [7]:
from datetime import datetime

# Generate a new timestamp for this run, or use an existing one
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# run_timestamp = "20260714_151745"  # Uncomment to reuse existing run

# Config
START_YEAR = "2023"
END_YEAR = "2026"
NETWORK_MODE = "full"  # Options: ego, full, global
CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

print(f"run_timestamp: {run_timestamp}")

run_timestamp: 20260715_155130


In [ ]:
# Step 1: Run cwts_export.py
import subprocess, sys, os

env = os.environ.copy()
env["START_YEAR"] = START_YEAR
env["END_YEAR"] = END_YEAR
env["NETWORK_MODE"] = NETWORK_MODE
env["RUN_TIMESTAMP"] = run_timestamp

print(f"Running cwts_export.py with run_timestamp={run_timestamp}...")
result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

In [3]:
# timestamp = "20260710_144411" # 2020-2026 full run
# timestamp = "20260618_130306"  # 2023-2026 full run
timestamp = "20260714_151745"  # tes run with new journals

In [5]:
# Step 2: Generate GPT cluster labels
from src import label_clusters

label_clusters.main(timestamp)

Loading from BigQuery (run_timestamp=20260714_151745)...
  Classification: ocean-tech-adv-analytics-c-tfs.scope_drift_raw.classification_raw_20260714_151745
  Pub metadata:   ocean-tech-adv-analytics-c-tfs.scope_drift_raw.pub_metadata_raw_20260714_151745
  Cit links:      ocean-tech-adv-analytics-c-tfs.scope_drift_raw.cit_links_raw_20260714_151745

Processing macro level...
  Fetching up to 250 titles per cluster...
Downloading: 100%|██████████|
  Loaded 1,750 titles across 7 clusters

--- Pass 1: Labelling macro (7 clusters) ---
  [1/7] cluster 0 (831,250 papers) → Cancer Research
  [2/7] cluster 1 (714,866 papers) → Nanomaterials And Photocatalysis
  [3/7] cluster 2 (579,756 papers) → Environmental Sustainability
  [4/7] cluster 3 (321,631 papers) → Machine Learning Applications
  [5/7] cluster 4 (253,424 papers) → Geotechnical Engineering
  [6/7] cluster 5 (179,196 papers) → Neuroscience And Mental Health
  [7/7] cluster 6 (63,596 papers) → Clinical Outcomes

  No duplicate labels f

In [9]:
# Step 3: Generate dashboard
import subprocess, sys, os

env = os.environ.copy()
env["RUN_TIMESTAMP"] = timestamp
env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

result = subprocess.run(
    [sys.executable, "src/build_unified_dashboard.py"],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("\nDashboard ready: output/combined_dashboard.html")


Downloading:   0%|          |
Downloading: 100%|██████████|





C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
15:51:48  ============================================================
15:51:48  Unified Dashboard Builder (BigQuery)
15:51:48  ============================================================
15:51:48  RUN_TIMESTAMP: 20260714_151745
15:51:48  Cluster level: macro
15:51:48  Output dir: C:\Users\sophie.wilson\Documents\scope-drift-model\output
15:51:48  
[LOAD] Loading data from BigQuery …
15:51:49         Project: ocean-tech-adv-analytics-c-tfs
15:51:49         Dataset: scope_drift_raw
15:51:49         Timestamp: 20260714_151745
15:51:49         Classification: ocean-tech-adv-analytics-c-tfs.scope_drift_raw.classification_raw_20260714_151745
15:51:49         Pub metadata: ocean-tech-adv-an

---
# Old Cells (Reference)

In [ ]:
# import subprocess, sys, os

# """
# This output is:
#  1. pubs.txt - Paper list for CWTS tool
#     Columns: int_id  paper identifier for cwts, core_pub (always 1 as not using core feature, but it has to be in)

#  2. cit_links.txt - Citation network edges
#     Columns:
#     int_id1 - citing paper,
#     int_id2 - cited paper,
#     weight - citation strength 0-2 higher= stronger
#     Note: Each edge appears twice (A→B and B→A) for undirected format
#         paper 5 cites Paper 12  →  row: 5, 12, 0.85
#         Paper 12 cites Paper 5  →  row: 12, 5, 0.85  (same edge, reversed)

#  3. pub_metadata.txt - Paper details lookup table
#     Columns: int_id, pub_id, is_frontiers, journal, date, title
#     - int_id: sequential CWTS ID (joins to classification.txt)
#     - pub_id: airak PublicationId (joins to BigQuery tables)
#  JOIN KEY: int_id links all files together, this is cwts identifier
# """
# print("ere")
# env = os.environ.copy()
# print("ere")
# env["START_YEAR"] = "2023"
# env["END_YEAR"] = "2026"
# env["NETWORK_MODE"] = "full"
# env["RUN_TIMESTAMP"] = run_timestamp  # Must be uppercase to match cwts_export.py
# print("ere")
# result = subprocess.run(
#     [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
# )
# print("ere")
# print(result.stdout)
# print(result.stderr)
# print("last")

In [ ]:
# import subprocess
# import datetime
# import os
# import pandas as pd

# # --- Parameters ---
# params = {
#     "largest_component_only": "true",
#     "iterations": "100",
#     "micro_resolution": "5e-4",
#     "micro_min_cluster_size": "1000",
#     "meso_resolution": "5e-6",
#     "meso_min_cluster_size": "5000",
#     "macro_resolution": "1e-6",
#     "macro_min_cluster_size": "20000",
# }


# input_files = {
#     "pubs": "cwts_output/pubs.txt",
#     "cit_links": "cwts_output/cit_links.txt",
#     "output": "cwts_output/classification.txt",
#     "jar": "publicationclassification.jar",
# }


# result = subprocess.run(
#     [
#         "java",
#         "-cp",
#         input_files["jar"],
#         "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
#         input_files["pubs"],
#         input_files["cit_links"],
#         input_files["output"],
#         params["largest_component_only"],
#         params["iterations"],
#         params["micro_resolution"],
#         params["micro_min_cluster_size"],
#         params["meso_resolution"],
#         params["meso_min_cluster_size"],
#         params["macro_resolution"],
#         params["macro_min_cluster_size"],
#     ],
#     capture_output=True,
#     text=True,
# )

# # --- Log ---
# os.makedirs("logs", exist_ok=True)
# log_path = f"logs/cwts_run_{run_timestamp}.log"

# with open(log_path, "w") as f:
#     f.write(f"CWTS Publication Classification Run\n")
#     f.write(f"{'='*50}\n")
#     f.write(f"Timestamp : {run_timestamp}\n\n")

#     f.write(f"Input Files\n{'-'*30}\n")
#     for k, v in input_files.items():
#         f.write(f"  {k:<20}: {v}\n")

#     f.write(f"\nParameters\n{'-'*30}\n")
#     for k, v in params.items():
#         f.write(f"  {k:<26}: {v}\n")

#     f.write(f"\nReturn Code: {result.returncode}\n")

#     f.write(f"\nSTDOUT\n{'-'*30}\n")
#     f.write(result.stdout or "(empty)\n")

#     f.write(f"\nSTDERR\n{'-'*30}\n")
#     f.write(result.stderr or "(empty)\n")

# print(f"Log written to: {log_path}")
# print(result.stdout)
# if result.stderr:
#     print(result.stderr)

# # Load classification.txt
# classification = pd.read_csv(
#     "cwts_output/classification.txt",
#     sep="\t",
#     header=None,
#     names=["int_id", "micro", "meso", "macro"],
# )

# # Upload to BigQuery
# BQ_DEST_PROJECT = "ocean-tech-adv-analytics-c-tfs"
# BQ_DEST_DATASET = "scope_drift_raw"


# # classification.to_gbq(
# #     f"{dataset}.classification_raw_{run_timestamp}",
# #     project_id=project,
# #     if_exists="replace",
# # )

# # Upload to BigQuery

# classification.to_gbq(
#     f"{BQ_DEST_DATASET}.classification_raw_{run_timestamp}",
#     project_id=BQ_DEST_PROJECT,
#     if_exists="replace",
# )
# print(f"  → BigQuery: {BQ_DEST_DATASET}.classification_raw_{run_timestamp}")

In [ ]:
# import pandas as pd


# df = pd.read_csv(
#     "cwts_output/classification.txt",
#     sep="\t",
#     header=None,
#     names=["pub_no", "micro", "meso", "macro"],
# )


# print(f"Total classified: {len(df):,}")


# for level in ["micro", "meso", "macro"]:

#     vc = df[level].value_counts()

#     print(f"\n{level.upper()}: {len(vc):,} clusters")

#     print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")

#     print(f"  Smallest: {vc.iloc[-1]:,}")

#     print(f"  Median  : {vc.median():.0f}")

### Labelling with GPT

In [ ]:
# timestamp = "20260710_144411" # 2020-2026 full run
# timestamp = "20260618_130306"  # 2023-2026 full run
# timestamp = "20260714_151745"  # tes run with new journals

In [ ]:
# import label_clusters

# # Run the script (reads from BigQuery using run_timestamp)
# label_clusters.main(timestamp)

## Labelling with taxonomy 
- this is not as good as raw GPT so for now im going to drop it and ill ask toby what to do later

In [ ]:
# import taxonomy_naming

# # Run taxonomy naming (reads from BigQuery using run_timestamp)
# df_out = taxonomy_naming.main(timestamp)

### looking at scope

In [ ]:
# import importlib
# import journal_scope
# from pathlib import Path
# import os

# # Override config variables
# journal_scope.SCOPE_LEVEL = "macro"
# journal_scope.SCOPE_THRESHOLD = 0.80
# journal_scope.MIN_PAPERS = 50
# journal_scope.USE_GPT = False
# journal_scope.OUTPUT_DIR = Path("cwts_output")
# # Data source - set on the module, not as local variables
# journal_scope.DATA_SOURCE = "bigquery"
# journal_scope.RUN_TIMESTAMP = timestamp
# journal_scope.TARGET_JOURNALS = [
#     "Frontiers in Immunology",
#     "Frontiers in Public Health",
#     "Frontiers in Medicine",
#     "Frontiers in Oncology",
#     "Frontiers in Psychology",
# ]
# #
# journal_scope.main()

### Generate Dashboard

In [ ]:
# import subprocess
# import os

# env = os.environ.copy()
# env["RUN_TIMESTAMP"] = timestamp
# env["CLUSTER_LEVEL"] = "macro"  # optional
# # env["JOURNALS"] = "Frontiers in Immunology,Frontiers in Oncology"  # optional override
# env["BUILD_ALL"] = "1"  # <-- Add this to regenerate dashboards
# result = subprocess.run(
#     ["python", "scripts/build_unified_dashboard.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )
# # Checking for errors as this is claude generated
# print("Return code:", result.returncode)
# print("STDOUT:", result.stdout)
# print("STDERR:", result.stderr)

In [ ]:
# import subprocess
# import sys
# import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the scope dashboard from local CWTS files
# # Uses: cwts_output/classification.txt, cwts_output/pub_metadata.txt, cwts_output/cit_links.txt
# # Outputs: output/scope_dashboard.html
# # Metadata pulled from BigQuery using run_timestamp
# print(timestamp)
# env = os.environ.copy()
# # env["CLUSTER_LEVEL"] = CLUSTER_LEVEL
# env["RUN_TIMESTAMP"] = timestamp  # Pass timestamp to fetch metadata from BigQuery
# env["DATA_SOURCE"] = "bigquery"

# result = subprocess.run(
#     [sys.executable, "scripts/build_dashboard_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/scope_dashboard.html")

In [ ]:
# import subprocess
# import sys
# import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the drift dashboard (JSD trends, heatmap, entropy changes)
# # Compares current cluster distribution vs baseline (2018-2020)
# # Outputs: output/drift_dashboard.html

# env = os.environ.copy()
# env["CLUSTER_LEVEL"] = CLUSTER_LEVEL
# env["RUN_TIMESTAMP"] = timestamp  # Pass timestamp to fetch metadata from BigQuery

# result = subprocess.run(
#     [sys.executable, "scripts/build_drift_dashboard_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/drift_dashboard.html")

In [ ]:
# import subprocess
# import sys
# import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the cluster bubble dashboard
# # Shows clusters as bubbles positioned by citation relationships
# # Outputs: output/cluster_bubbles.html

# env = os.environ.copy()
# env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

# result = subprocess.run(
#     [sys.executable, "scripts/build_cluster_bubbles_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/cluster_bubbles.html")

In [ ]:
# import subprocess
# import sys

# # Build the clusters hierarchy dashboard
# # Shows macro/meso/micro clusters with GPT labels
# # Outputs: output/clusters.html

# result = subprocess.run(
#     [sys.executable, "scripts/build_clusters_from_cwts.py"],
#     capture_output=True,
#     text=True,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/clusters.html")